### Embedding Geometry Sandbox

Notebook to explore embedding geometry: load or compute embeddings, estimate metrics similar to `embedding_geometry.py`, and approximate volume/capacity using the formulas from `th_bounds_estimation.ipynb`.

### Load embeddings
- `load_embeddings_from_disk`: pick a saved `.npy` embedding matrix, generated by  `generate_embeddings.py`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SAMPLED_EMB_DIR = Path('data/embeddings_samples')
RECTS_DIR = Path('data/rects_inf')
MEDIAN_ITERATIONS_PATH = Path('figures/convolutions/median_by_iteration.txt')
SAMPLED_LENGTHS = [4, 8,16,32,64,128,256,512,1024, 2048, 4096]
DEFAULT_P = 2 # dimension for log_frac_cone

MODELS = [
    # 'pythia-160m',
    # 'pythia-410m',
    # 'pythia-1b',
    'pythia-1.4b',
    'pythia-2.8b',
    'pythia-6.9b',
    # 'Qwen2.5-0.5B',
    # 'Qwen2.5-1.5B',
    # 'Llama-3.2-1B',
    # 'gemma-3-270m',
]
def load_embeddings_from_disk(model, lengths=SAMPLED_LENGTHS) -> np.ndarray:
    sample_emb = []
    for length in lengths:
        sample_path = SAMPLED_EMB_DIR / model / f'embedding_{length}.npy'
        if not sample_path.exists():
            raise FileNotFoundError(f'Expected sample file at {sample_path}')
        sample_emb_length = np.load(sample_path)
        sample_emb.append(sample_emb_length)
    
    return np.concatenate(sample_emb, axis=0)

### Estimate upper bounds and geometry

In [ ]:
# Cumulative rectangle + cone log-volumes for all models
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy import arccos, log, exp
from math import sin, cos
from scipy.special import betainc, beta, gammaln

CSV_PATH = Path('models_list.csv')
model_table = pd.read_csv(CSV_PATH).replace('--', pd.NA)
model_table['name'] = model_table['Link to HF'].map(lambda x: x.split("/")[-1])

def l1_l2_linf_radii(emb: np.ndarray):
    r1 = float(np.max(np.linalg.norm(emb, ord=1, axis=1)))
    r2 = float(np.max(np.linalg.norm(emb, ord=2, axis=1)))
    rinf = float(np.max(np.linalg.norm(emb, ord=np.inf, axis=1)))
    return r1, r2, rinf

def average_pairwise_cosine(emb: np.ndarray):
    norms = np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12
    X = emb / norms
    sim = X @ X.T
    triu = sim[np.triu_indices(sim.shape[0], k=1)]
    return float(triu.mean()), float(triu.min())

def th_slope(r, d, v, eps):
    """Theoretical slope using L_inf radius."""
    return float((d * log(1 + 2 * r / eps)) / log(v))

def log_frac_cone(min_cos, d, p=2):
    cos_sim = float(np.clip(min_cos, -1 + 1e-8, 1 - 1e-8))
    theta = arccos(cos_sim)
    a = gammaln((p + d) / p) - gammaln((p + 1) / p) - gammaln((p + d - 1) / p)  # no r term
    b1 = log(2) - log(d) + (d - 1) * log(sin(theta / 2)) + log(cos(theta / 2))
    b2 = log(beta(1 / 2, (d + 1) / 2)) + log(1 - betainc(1 / 2, (d + 1) / 2, cos(theta / 2) ** 2))
    m = max(b1, b2)
    return float(a + m + log(exp(b1 - m) + exp(b2 - m)))

def th_slope_with_cone(r, min_cos, d, v, eps):
    """Theoretical slope including cone fraction. """
    return float((d * log(1 + 2 * r / eps) + log_frac_cone(min_cos, d)) / log(v))

def th_slope_rect(rect, d, v, eps):
    """Theoretical slope using the L_inf rectangle."""
    return float(sum([log(1 + ri / eps) for ri in rect]) / log(v))

def stats_from_models(model, d, v, slope_exp, lengths=SAMPLED_LENGTHS):
    embs = load_embeddings_from_disk(model, lengths=lengths)
    print(f"  - lengths: {lengths}, embeddings shape: {embs.shape}")
    r1, r2, rinf = l1_l2_linf_radii(embs)
    # rect = np.max(np.abs(embs), axis=0) #- np.min(embs, axis=0)
    rect = np.max(embs, axis=0) - np.min(embs, axis=0)
    
    avg_cos, min_cos = average_pairwise_cosine(embs)
    epsilon = 2 ** ( log(rinf)/log(2) - 11)
    # print(d, rect.shape)
    slope =  th_slope(rinf, d, v, epsilon)
    slope_cone =  th_slope_with_cone(rinf, min_cos, d, v, epsilon)
    slope_rect =  th_slope_rect(rect, d, v, epsilon)
    # print(rinf, np.max(rect))
    print(f"    - Theoretical slopes: slope={slope}, slope_cone={slope_cone}, slope_rect={slope_rect}")
    return {
        'model': model,
        'lengths_max': lengths[-1],
        'd': d,
        'v': v,
        'r1': r1,
        'r2': r2,
        'rinf': rinf,
        'min_cos': min_cos,
        'avg_cos': avg_cos,
        'theta': float(arccos(min_cos)),
        'epsilon': epsilon,
        'rect_mean': float(np.mean(rect)),
        'th_slope':slope,     
        'th_slope_cone':slope_cone,
        'th_slope_rect':slope_rect,
        'slope_exp':slope_exp,
        'ratio_th': slope / slope_exp,
        'ratio_th_cone': slope_cone / slope_exp,
        'ratio_th_rect': slope_rect / slope_exp,
    }

### Plot influence of \ell on upper bounds 

In [ ]:
embedding_stats = {}
for model in MODELS:
    print(model)
    embedding_stats[model] = []

    d = int(model_table.loc[model_table['name'] == model, 'Input hidden size'])
    v = int(model_table.loc[model_table['name'] == model, 'Vocabulary size'])
    slope_exp = float(model_table.loc[model_table['name'] == model, 'Slope exp'])

    print(f"Summarizing geometry for model {model}")
    for k in range(1, len(SAMPLED_LENGTHS)+1):
        lengths = SAMPLED_LENGTHS[:k]

        stats = stats_from_models(model, d, v, slope_exp, lengths=lengths)
        embedding_stats[model].append(stats)

In [ ]:
models = MODELS
ncols = 3
nrows = int(np.ceil(len(models) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False, sharex=False)
for ax, model in zip(axes.ravel(), models):
    ax.plot([x['lengths_max'] for x in embedding_stats[model]][:-2], [x['th_slope'] for x in embedding_stats[model]][:-2]    , linestyle='-', marker='o', markersize=4, label='Ball')
    ax.plot([x['lengths_max'] for x in embedding_stats[model]][:-2], [x['th_slope_rect'] for x in embedding_stats[model]][:-2], linestyle=':', marker='o', markersize=4, label='Ellipsoid')
    ax.plot([x['lengths_max'] for x in embedding_stats[model]][:-2], [x['th_slope_cone'] for x in embedding_stats[model]][:-2], linestyle='--', marker='o', markersize=4, label='Cone')
    ax.set_xlabel(r"max sampled input length $\ell$ (tokens)")
    ax.set_ylabel(r"upper bound on slope $C$")
    ax.legend(fontsize=10)
    ax.set_title(model.capitalize(), fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')
    ax.locator_params(axis='y', nbins=2) 
    ax.locator_params(axis='x', nbins=3) 

for ax in axes.ravel()[len(models):]:
    ax.axis("off")

fig.tight_layout()
fig.savefig("theoretical_slopes_archi_big.pdf", dpi=200)
plt.show()

### Estimate statistics on models for all lengths
The helpers mirror those in `embedding_geometry.py`. `average_pairwise_cosine` is quadratic in the number of samples, so a down-sample option is provided for large matrices.

In [ ]:

embedding_stats = []
for model in MODELS:
    print(model)
    d = int(model_table.loc[model_table['name'] == model, 'Input hidden size'])
    v = int(model_table.loc[model_table['name'] == model, 'Vocabulary size'])
    slope_exp = float(model_table.loc[model_table['name'] == model, 'Slope exp'])

    print(f"Summarizing geometry for model {model}")
    stats = stats_from_models(model, d, v, slope_exp, lengths=SAMPLED_LENGTHS)
    embedding_stats.append(stats)

df_results = pd.DataFrame(embedding_stats)

df_results

In [ ]:
df_results[['model','d','v','rinf','theta','epsilon','ratio_th','ratio_th_cone','ratio_th_rect']]

| model | d | v | rinf | theta | epsilon | ratio_th | ratio_th_cone | ratio_th_rect |
|---|---|---|---|---|---|---|---|---|
| 0| pythia-1.4b| 2048| 50304| 59.90625| 2.219370| 0.029251| 6.019802| 5.938543| 4.316811 |
| 1| pythia-2.8b| 2560| 50304| 52.21875| 2.159577| 0.025497| 9.939214| 9.787080| 7.161607 |
| 2| pythia-6.9b| 4096| 50304| 50.96875| 2.213061| 0.024887| 11.099110| 10.948394| 7.704780 |

### Create rectangle coordinate txt for each model 

In [ ]:
# Save rectangle bounds for each model.
RECTS_DIR.mkdir(parents=True, exist_ok=True)

for model in MODELS:
    embs = load_embeddings_from_disk(model, lengths=SAMPLED_LENGTHS)
    rect_min = embs.min(axis=0)
    rect_max = embs.max(axis=0)

    output_path = RECTS_DIR / f'{model}.txt'
    with output_path.open('w', encoding='utf-8') as f:
        for min_val, max_val in zip(rect_min, rect_max):
            f.write(f"{min_val:.6f} {max_val:.6f}
")

    print(f"Saved rectangle for {model} to {output_path}")

### AFTER DISTRIBUTION REFINEMENT -- Estimate median convolution slope for each model 

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path
from math import log
import numpy as np
path = MEDIAN_ITERATIONS_PATH

# Collect data per model
by_model = defaultdict(list)
with path.open(newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        model = row["model"]
        x = float(row["iteration"])
        y = float(row["median"])
        by_model[model].append((x, y))

# Simple linear regression: slope = cov(x,y)/var(x)
slopes = []
for model, pts in by_model.items():
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    x = np.array(xs)
    y = np.log(np.array(ys))
    print(y[:5])
    A = np.vstack([x, np.ones(len(x))]).T
    m, c = np.linalg.lstsq(A, y, rcond=None)[0]
    r2 = 1 - (np.sum((y - (m*x + c))**2) / np.sum((y - y.mean())**2))
    slopes.append((model, m, c, r2))

vs = [50304, 50304, 50304, 151936, 151936,262144,128256]
slopes.sort(key=lambda t: t[0])
print("Model\t-log(slope)\t-log(vocab_size)\tR^2")
for model, m, b, n in slopes:
    print(f"{model}\t{m:.2f}\t{-log(vs.pop(0)):.2f}\t{r2:.2f}")

| Model | Slope Conv | Slope Vocab
| - | - | -
| EleutherAI/pythia-160m	| -12.87 |	-10.83
| EleutherAI/pythia-1b	    | -14.71 |	-10.83
| EleutherAI/pythia-410m	| -14.54 |	-10.83
| Qwen/Qwen2.5-0.5B	        | -16.50 |	-11.93
| Qwen/Qwen2.5-1.5B	        | -16.87 |	-11.93
| google/gemma-3-270m	    | -13.03 |	-12.48
| meta-llama/Llama-3.2-1B	| -15.79 |	-11.76


In [ ]:
df_results['log_slope_conv'] = pd.Series([ -12.87, -14.71, -14.54, -16.50, -16.87, -13.03, -15.79 ])

def th_slope_conv_rect(slope_rect, v, log_slope_conv):
    """Theoretical slope using rectangle with convolution."""
    return float(slope_rect * log(v) / log_slope_conv)

df_results['th_slope_conv_rect'] = df_results.apply(
    lambda row: th_slope_conv_rect(row['th_slope_rect'], row['v'], -row['log_slope_conv']), 
    axis=1)
df_results['ratio_conv_rect'] = df_results['th_slope_conv_rect'] / df_results['slope_exp']

In [ ]:
df_results[['model', 'ratio_th','ratio_th_cone','ratio_th_rect','ratio_conv_rect']]